# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jenilrupareliya5150-bit/FlyRankAi-ml-Track/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

==> 1. Question

### Research question

**Which content pages should be reviewed first for improvement based on observable search-performance signals?**

The goal is to build a leakage-safe ranking model that helps prioritize content for human review.

The model will use historical and observable signals such as Google Search Console impressions, clicks, and average position. It will predict whether the page's **next-day Google Search Console impressions** will reach a selected performance threshold.

This project is intended for decision support, not automatic publishing or guaranteed SEO improvement.

In [38]:

import numpy as np
import pandas as pd
import seaborn as sns

from datasets import load_dataset

from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

from IPython.display import display

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [13]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [14]:
#load wharehouse dataset
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

print("Real FlyRank warehouse connected successfully.")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Real FlyRank warehouse connected successfully.


In [17]:
# Read rows from the real warehouse

MAX_ROWS = 500_000

rows = []

for i, row in enumerate(ds):
    rows.append(row)

    if i + 1 >= MAX_ROWS:
        break

real_df = pd.DataFrame(rows)

print("Real dataset shape:", real_df.shape)
print("\nColumns:")
print(real_df.columns.tolist())

display(real_df.sample())

Real dataset shape: (500000, 30)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
103688,2025-03-17,client_9958f0a7ae1df715,content_278af14d7e0781d1,True,True,True,False,3,0,44.0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
#  data inspection

print("Rows:", len(real_df))
print("Columns:", len(real_df.columns))

print("\nData types:")
display(real_df.dtypes)

print("\nMissing values:")
display(
    real_df.isna().sum().sort_values(ascending=False).head(15)
)

Rows: 500000
Columns: 30

Data types:


,0
report_date,object
client_hash_id,object
content_hash_id,object
client_has_gsc,bool
client_has_ga4,bool
gsc_data_available,bool
ga4_data_available,bool
gsc_impressions,int64
gsc_clicks,int64
gsc_sum_position,float64



Missing values:


,0
gsc_avg_position,1
gsc_sum_position,1
content_hash_id,0
client_has_gsc,0
report_date,0
client_hash_id,0
gsc_data_available,0
client_has_ga4,0
gsc_impressions,0
ga4_data_available,0


In [19]:
real_df.shape

(500000, 30)

In [20]:
df=real_df.copy()

In [23]:
cols=[real_df.columns.tolist()]

In [35]:
null_cols=[]
for i in cols:
  null_cols.append([real_df[i],real_df[i].isna().sum()])

In [36]:
null_cols

[[       report_date           client_hash_id           content_hash_id  \
  0       2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
  1       2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
  2       2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   
  3       2025-01-27  client_9958f0a7ae1df715  content_c899aef92518c714   
  4       2025-01-27  client_9958f0a7ae1df715  content_c7c1d2e68d9d0964   
  ...            ...                      ...                       ...   
  499995  2025-04-07  client_73cda7b4e4f265ea  content_4e39b033728328dd   
  499996  2025-04-07  client_73cda7b4e4f265ea  content_8204e1531c126b67   
  499997  2025-04-07  client_73cda7b4e4f265ea  content_8e3b05744d555c4f   
  499998  2025-04-07  client_73cda7b4e4f265ea  content_79df4b8a74d75a1a   
  499999  2025-04-07  client_73cda7b4e4f265ea  content_d4b7791aa9c13054   
  
          client_has_gsc  client_has_ga4  gsc_data_available  \
  0                 True         

In [39]:
real_df.describe()

,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,ga4_total_engagement_sec,sessions_organic,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
count,500000.000000,500000.000000,499999.000000,499999.000000,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0,...,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0,500000.0
mean,25.817892,0.163506,515.098558,28.917954,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
std,74.713565,0.942207,1021.818140,23.322007,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
min,1.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,3.000000,0.000000,74.000000,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,9.000000,0.000000,211.000000,21.333333,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
75%,24.000000,0.000000,548.000000,44.888889,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,7889.000000,173.000000,193998.000000,308.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## 2. Data

### Dataset

This project uses the FlyRank pseudonymized warehouse data.

The main modeling table is:

`fact_content_daily_performance`

Each row represents the daily performance of a content item for a client.

### Working dataset

For this capstone notebook, we use a 500,000-row working sample streamed from the real warehouse.

The actual date range present in this working dataset is:

**2025-01-27 to 2025-04-30**

The date range is verified directly from the loaded data.

### Main signals

The dataset contains historical search and engagement signals including:

- Google Search Console impressions
- Google Search Console clicks
- Average and summed search position
- Google Analytics 4 pageviews and sessions
- Organic, direct, referral, social and paid sessions
- AI-related traffic signals
- Scroll events

### Exclusions

Identifiers such as `client_hash_id` and `content_hash_id` are used to construct observations and joins but are not used as predictive features.

Future-performance information is also excluded from the model inputs. Future observations are used only to construct the prediction target.

This is necessary to prevent data leakage.

### Public-safe handling

The dataset uses pseudonymized identifiers. No client names, domains, URLs, private queries, credentials, or other identifying information will be included in the final public research paper.

In [43]:
# Section 2--> Data verification

print("Dataset shape:", real_df.shape)

print("\nDate range:")
print("Minimum date:", real_df["report_date"].min())
print("Maximum date:", real_df["report_date"].max())

print("\nNumber of columns:", len(real_df.columns))

print("\nColumns:")
for i, col in enumerate(real_df.columns, start=1):
    print(f"{i}. {col}")

Dataset shape: (500000, 30)

Date range:
Minimum date: 2025-01-27
Maximum date: 2025-04-30

Number of columns: 30

Columns:
1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events


In [44]:
# Check the grain / unique entities

print("Unique clients:", real_df["client_hash_id"].nunique())
print("Unique content items:", real_df["content_hash_id"].nunique())
print("Unique dates:", real_df["report_date"].nunique())

Unique clients: 4
Unique content items: 14259
Unique dates: 94


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

===> Methodology

### Modeling approach

The modeling task is treated as a supervised binary classification problem.

For each content item, information available at the current observation is used to predict a future performance outcome.

### Target definition

We define the target using the next observed daily Google Search Console impressions value.

A row receives:

- `target = 1` if the next observed GSC impressions reach the selected threshold.
- `target = 0` otherwise.

The future GSC impressions value is used only to construct the target and is never included as a model feature.

### Features

Candidate features are taken from the current observation and include available search and engagement signals such as:

- GSC impressions
- GSC clicks
- GSC summed position
- GSC average position
- GA4 pageviews
- GA4 sessions
- other available current-period performance signals

Identifiers are excluded from the predictive feature set.

### Leakage control

Future-performance fields are excluded from the model inputs.

The target is created before modeling, but the future observation used to create the target is never provided to the model as a feature.

This ensures that the model only uses information that would have been available at prediction time.

### Train/test validation

The observations are sorted chronologically.

The earlier observations are used for training and the later observations are held out for testing.

This time-aware split is used instead of randomly mixing observations so that future observations do not enter the training data.

### Baseline

A simple majority/prior-probability baseline is used as a transparent reference point.

The Random Forest classifier is then evaluated against this baseline.

### Evaluation metrics

Because the intended output is a ranked review queue, we evaluate:

- ROC-AUC
- Average Precision
- Precision
- Recall
- F1 score
- Accuracy
- Precision@K

Precision@K is particularly relevant because the practical goal is to identify a limited number of high-priority pages for human review.

### Interpretation

The model is used for prioritization rather than causal inference. A high prediction score indicates similarity to historical observations associated with the positive target; it does not prove that refreshing the page will cause better performance.

In [45]:

df["report_date"] = pd.to_datetime(df["report_date"])#it converts any dtype into standerd date time formate

df = df.sort_values(
    ["client_hash_id", "content_hash_id", "report_date"]
).reset_index(drop=True)

print("Prepared dataset shape:", df.shape)
print("Date range:")
print(df["report_date"].min(), "to", df["report_date"].max())

Prepared dataset shape: (500000, 30)
Date range:
2025-01-27 00:00:00 to 2025-04-30 00:00:00


In [50]:
df.shape

(500000, 33)

In [46]:
# Create next-day observations

df["next_date"] = (
    df["report_date"] + pd.Timedelta(days=1)
)

next_day = df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions"
    ]
].copy()

next_day = next_day.rename(
    columns={
        "report_date": "next_date",
        "gsc_impressions": "next_gsc_impressions"
    }
)

df = df.merge(
    next_day,
    on=[
        "client_hash_id",
        "content_hash_id",
        "next_date"
    ],
    how="left"
)

df["next_day_observed"] = (
    df["next_gsc_impressions"].notna()
)

model_df = df[
    df["next_day_observed"]
].copy()

print("Original rows:", len(df))
print("Rows with next-day observation:", len(model_df))

Original rows: 500000
Rows with next-day observation: 427644


In [51]:
df.sample(5)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,next_date,next_gsc_impressions,next_day_observed
314106,2025-04-04,client_73cda7b4e4f265ea,content_fa9aa890f96d2c1b,True,True,True,False,3,0,119.0,...,0,0,0,0,0,0,0,2025-04-05,3.0,True
198325,2025-04-04,client_73cda7b4e4f265ea,content_a146747a187403d7,True,True,True,False,2,0,31.0,...,0,0,0,0,0,0,0,2025-04-05,NaN,False
23305,2025-03-23,client_73cda7b4e4f265ea,content_159535612a8a07f5,True,True,True,False,10,0,112.0,...,0,0,0,0,0,0,0,2025-03-24,9.0,True
197125,2025-02-22,client_73cda7b4e4f265ea,content_a03fe1c3c1e311ba,True,True,True,False,18,0,187.0,...,0,0,0,0,0,0,0,2025-02-23,23.0,True
92663,2025-04-28,client_73cda7b4e4f265ea,content_4d0d5f711ce8fbd7,True,True,True,False,1,0,11.0,...,0,0,0,0,0,0,0,2025-04-29,NaN,False


In [52]:
# Create binary target


target_threshold = model_df[
    "next_gsc_impressions"
].quantile(0.70)

model_df["target"] = (
    model_df["next_gsc_impressions"]
    >= target_threshold
).astype(int)#convert into binary target

print("Target threshold:", target_threshold)

print("\nTarget distribution:")
print(model_df["target"].value_counts())

print("\nTarget proportion:")
print(model_df["target"].value_counts(normalize=True))

Target threshold: 21.0

Target distribution:
target
0    294190
1    133454
Name: count, dtype: int64

Target proportion:
target
0    0.687932
1    0.312068
Name: proportion, dtype: float64


In [54]:
model_df["target"]

,target
0,0
1,0
2,0
3,0
4,0
...,...
499994,0
499995,1
499996,0
499997,0


In [55]:

# Select features

candidate_features = [
    col for col in model_df.columns
    if col not in [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "next_date",
        "next_gsc_impressions",
        "next_day_observed",
        "target"
    ]
]

print("Number of candidate features:", len(candidate_features))
print("\nCandidate features:")
print(candidate_features)

Number of candidate features: 27

Candidate features:
['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


In [56]:

# Remove constant /missing features

usable_features = []

for col in candidate_features:
    if model_df[col].notna().sum() == 0:
        continue

    if model_df[col].nunique(dropna=True) <= 1:
        continue

    usable_features.append(col)

print("Usable features:", len(usable_features))
print("\nRemoved features:")
print(set(candidate_features) - set(usable_features))

Usable features: 4

Removed features:
{'scroll_events', 'ga4_users', 'gsc_data_available', 'ga4_total_engagement_sec', 'ga4_pageviews', 'sessions_ai', 'ai_gemini', 'sessions_social', 'ai_meta', 'client_has_ga4', 'sessions_paid', 'ga4_engaged_sessions', 'ai_claude', 'ai_perplexity', 'sessions_referral', 'ga4_data_available', 'ai_copilot', 'ai_other', 'sessions_organic', 'ga4_sessions', 'client_has_gsc', 'ai_chatgpt', 'sessions_direct'}


In [57]:
#create a model input
X = model_df[usable_features].copy()
y = model_df["target"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

X shape: (427644, 4)
y shape: (427644,)

Target distribution:
target
0    294190
1    133454
Name: count, dtype: int64


In [58]:
#handling the issing values
missing_counts = X.isna().sum()

print("Columns containing missing values:")
display(
    missing_counts[missing_counts > 0]
    .sort_values(ascending=False)
)

Columns containing missing values:


,0
gsc_sum_position,1
gsc_avg_position,1


### Time-based train/test split

The data is split chronologically rather than randomly.

Earlier observations are used for training, while later observations are held out for testing. This better represents the real prediction setting, where the model is trained on historical observations and then applied to future
observations.

EARLIER DATA ==>train              
LATER DATA==>test
                               
         
                 

In [59]:
model_df = model_df.sort_values("report_date").reset_index(drop=True)#SORT THE DATASET ON THE BASIS OF THE DATE

split_date = model_df["report_date"].quantile(0.80)

train_df = model_df[
    model_df["report_date"] < split_date
].copy()#SELECT THE EARLIER DATA FOR TRAINNING

test_df = model_df[
    model_df["report_date"] >= split_date
].copy()

print("Split date:", split_date)

print("\nTraining:")
print("Rows:", len(train_df))
print("Date range:", train_df["report_date"].min(), "to", train_df["report_date"].max())

print("\nTesting:")
print("Rows:", len(test_df))
print("Date range:", test_df["report_date"].min(), "to", test_df["report_date"].max())

Split date: 2025-04-17 00:00:00

Training:
Rows: 336579
Date range: 2025-01-27 00:00:00 to 2025-04-16 00:00:00

Testing:
Rows: 91065
Date range: 2025-04-17 00:00:00 to 2025-04-29 00:00:00


In [62]:
# Final train/test feature matrices

X_train = train_df[usable_features].copy()
y_train = train_df["target"].copy()

X_test = test_df[usable_features].copy()
y_test = test_df["target"].copy()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (336579, 4)
y_train: (336579,)
X_test: (91065, 4)
y_test: (91065,)


In [63]:
# Median imputation using training data

train_medians = X_train.median(numeric_only=True)

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print("Remaining missing values in X_train:", X_train.isna().sum().sum())
print("Remaining missing values in X_test:", X_test.isna().sum().sum())

Remaining missing values in X_train: 0
Remaining missing values in X_test: 0


In [64]:
# Leakage audit

leakage_columns = [
    "next_gsc_impressions",
    "next_date",
    "next_day_observed",
    "target"
]

leakage_in_features = [
    col for col in leakage_columns
    if col in X_train.columns
]

print("Potential leakage columns found in features:")
print(leakage_in_features)

print("\nFuture GSC impressions in features:",
      "next_gsc_impressions" in X_train.columns)

print("\nNumber of final features:", len(X_train.columns))

Potential leakage columns found in features:
[]

Future GSC impressions in features: False

Number of final features: 4


In [67]:
# Methodology summary

print("<===== METHODOLOGY CHECK =====>")

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Number of features:", X_train.shape[1])

print(
    "Training positive rate:",
    round(y_train.mean(), 4)
)

print(
    "Testing positive rate:",
    round(y_test.mean(), 4)
)

print(
    "Leakage present:",
    len(leakage_in_features) > 0
)

print(
    "Train end date:",
    train_df["report_date"].max()
)

print(
    "Test start date:",
    test_df["report_date"].min()
)

<===== METHODOLOGY CHECK =====>
Training rows: 336579
Testing rows: 91065
Number of features: 4
Training positive rate: 0.3029
Testing positive rate: 0.3461
Leakage present: False
Train end date: 2025-04-16 00:00:00
Test start date: 2025-04-17 00:00:00


REAL DATA

        ↓

Create next_gsc_impressions
        
        ↓


Create target (y)
        ↓


Separate:


     ├── X = current-day features ONLY
     └── y = future target
      
       ↓

Time split
       
       
       ↓
Train / Test

1.   *List item*

1.   List item

*   List item
*   List item


2.   List item


2.   List item



## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

==>. Baseline and Model

### 4.1 Baseline

A simple baseline is used as a reference point before applying a machine-learning model.

The baseline predicts the majority class for every test observation. This provides a simple benchmark that the Random Forest model must outperform.

In [68]:
#  Baseline Model

baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
baseline_prob = baseline.predict_proba(X_test)[:, 1]

print("Baseline created successfully.")

Baseline created successfully.


The DummyClassifier in scikit-learn is a simple, non-parametric classifier used primarily as a baseline for comparison with more sophisticated classifiers. It doesn't learn any patterns from the data; instead, it makes predictions using simple rules or heuristics.



In [69]:
# Baseline evaluation

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_pred,
    zero_division=0
)

baseline_auc = roc_auc_score(
    y_test,
    baseline_prob
)

baseline_ap = average_precision_score(
    y_test,
    baseline_prob
)

print("Baseline Results")
print("----------------")
print("Accuracy:", round(baseline_accuracy, 4))
print("Precision:", round(baseline_precision, 4))
print("Recall:", round(baseline_recall, 4))
print("F1:", round(baseline_f1, 4))
print("ROC-AUC:", round(baseline_auc, 4))
print("Average Precision:", round(baseline_ap, 4))

Baseline Results
----------------
Accuracy: 0.6539
Precision: 0.0
Recall: 0.0
F1: 0.0
ROC-AUC: 0.5
Average Precision: 0.3461


### 4.3 Random Forest

A Random Forest classifier is trained using the current-period features.

The model is selected because it can capture nonlinear relationships and interactions between search and engagement signals while remaining relatively interpretable through feature importance.

The model is trained only on the historical training period and evaluated on the later held-out period.

In [70]:

# train Random Forest


rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully.")

Random Forest trained successfully.


In [71]:
#prediction

rf_pred = rf_model.predict(X_test)

rf_prob = rf_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated.")
print("Number of test predictions:", len(rf_pred))

Predictions generated.
Number of test predictions: 91065


In [72]:
# Random Forest evaluation


rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

rf_precision = precision_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_recall = recall_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_f1 = f1_score(
    y_test,
    rf_pred,
    zero_division=0
)

rf_auc = roc_auc_score(
    y_test,
    rf_prob
)

rf_ap = average_precision_score(
    y_test,
    rf_prob
)

print("Random Forest Results")
print("---------------------")
print("Accuracy:", round(rf_accuracy, 4))
print("Precision:", round(rf_precision, 4))
print("Recall:", round(rf_recall, 4))
print("F1:", round(rf_f1, 4))
print("ROC-AUC:", round(rf_auc, 4))
print("Average Precision:", round(rf_ap, 4))

Random Forest Results
---------------------
Accuracy: 0.8819
Precision: 0.8624
Recall: 0.7836
F1: 0.8211
ROC-AUC: 0.9276
Average Precision: 0.8954


In [73]:
# Precision@K

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1]
    top_k = order[:k]

    return np.mean(
        np.asarray(y_true)[top_k]
    )


for k in [20, 50, 100]:
    score = precision_at_k(
        y_test.values,
        rf_prob,
        k
    )

    print(
        f"Precision@{k}: {score:.4f}"
    )

Precision@20: 0.9000
Precision@50: 0.9400
Precision@100: 0.9600


In [74]:
# Model comparison

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Average Precision"
    ],
    "Baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_auc,
        baseline_ap
    ],
    "Random Forest": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1,
        rf_auc,
        rf_ap
    ]
})

comparison["Baseline"] = comparison["Baseline"].round(4)
comparison["Random Forest"] = comparison["Random Forest"].round(4)

display(comparison)

,Metric,Baseline,Random Forest
0,Accuracy,0.6539,0.8819
1,Precision,0.0000,0.8624
2,Recall,0.0000,0.7836
3,F1,0.0000,0.8211
4,ROC-AUC,0.5000,0.9276
5,Average Precision,0.3461,0.8954


In [75]:
# Feature importance

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(
    feature_importance.head(15)
)

,feature,importance
0,gsc_impressions,0.627016
1,gsc_sum_position,0.218584
2,gsc_avg_position,0.129306
3,gsc_clicks,0.025094


## 5. Limitations

*What this work cannot claim.*

## 5. Limitations

This analysis has several limitations that should be considered when interpreting the results.

### Observational data

The dataset contains historical observations of content performance. Therefore, the model identifies patterns associated with future search performance, but it does not prove that changing a particular content feature will cause impressions to increase.

### Target definition

The target is based on whether the next observed day's Google Search Console impressions cross the selected threshold. This is a useful proxy for prioritization, but it does not represent long-term SEO success or overall content quality.

### Data availability

The model can only use information that is available in the dataset at prediction time. Some GA4-related fields contain little or no variation in this dataset, which limits their usefulness for the model.

### Time-based validation

The model was evaluated using an earlier-period training set and a later-period test set. This better represents the intended future-use scenario, but performance may change when applied to a different time period.

### Class imbalance

The positive and negative target classes are not perfectly balanced. Therefore, accuracy alone is not sufficient to judge model quality. Ranking-oriented metrics such as Precision@50 and Average Precision are more relevant to the intended prioritization task.

### Generalization

The results are based on the available FlyRank warehouse data and its observed clients and content. The model may not perform equally well on other websites, industries, or future search environments.

### Feature limitations

Although multiple search and engagement signals were available, several features had little or no variation and therefore contributed little to the Random Forest model. The strongest feature importance was concentrated in Google Search Console impressions, position, and clicks.

### No causal claim

The model should be treated as a decision-support and prioritization tool. It should not be interpreted as evidence that a specific content refresh, SEO action, or other intervention will directly cause an improvement in search performance.

### Human review

The model provides a ranking signal for deciding which pages may deserve attention first. Final decisions should still consider content relevance, business context, search intent, and human review.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

===>. Ranked Recommendations

The Random Forest model can be used to prioritize pages for human review.

Pages are ranked by the model's predicted probability of reaching the selected future-impression threshold. A higher score means that the page is a stronger candidate for review based on the observed signals available at prediction time.

The recommendations are decision-support outputs, not guarantees that a refresh will improve performance.

Each recommendation includes:
- content identifier
- predicted score
- confidence level
- suggested action
- important observed signals

The recommended action is to review the highest-ranked pages first and determine whether a refresh, expansion, protection, or monitoring action is appropriate.

In [76]:
# 6. Ranked Recommendations

# Get prediction probabilities from the final Random Forest
test_scores = rf_model.predict_proba(X_test)[:, 1]

# Create recommendation dataframe from the test set
recommendations = test_df[
    ["client_hash_id", "content_hash_id", "report_date"]
].copy()

recommendations["model_score"] = test_scores

# Rank pages from highest to lowest model score
recommendations = recommendations.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

# Add rank
recommendations["rank"] = recommendations.index + 1

# Confidence label
def confidence_label(score):
    if score >= 0.80:
        return "High"
    elif score >= 0.60:
        return "Medium"
    else:
        return "Low"

recommendations["confidence"] = recommendations["model_score"].apply(
    confidence_label
)

# Suggested action
recommendations["suggested_action"] = "Review"

# Show top 50 recommendations
top_50 = recommendations.head(50).copy()

display(
    top_50[
        [
            "rank",
            "content_hash_id",
            "report_date",
            "model_score",
            "confidence",
            "suggested_action"
        ]
    ]
)

,rank,content_hash_id,report_date,model_score,confidence,suggested_action
0,1,content_88d500a55bd9b986,2025-04-29,1.0,High,Review
1,2,content_3fc3b8238a5b1ddf,2025-04-29,1.0,High,Review
2,3,content_3fa89de2020a2373,2025-04-29,1.0,High,Review
3,4,content_6d1a5f4347b19979,2025-04-29,1.0,High,Review
4,5,content_f18b70df51904723,2025-04-20,1.0,High,Review
5,6,content_f0f0f41959f01bb6,2025-04-17,1.0,High,Review
6,7,content_04441d837222c714,2025-04-17,1.0,High,Review
7,8,content_f0d9fb91e7681e85,2025-04-17,1.0,High,Review
8,9,content_05554cfab3e9cec8,2025-04-17,1.0,High,Review
9,10,content_9d8204e19b49515e,2025-04-20,1.0,High,Review


In [91]:
print("Ranked recommendation queue")
print("--------------------------------")
print("Total test pages:", len(recommendations))
print("Top 50 pages selected:", len(top_50))

print("\nConfidence distribution:")
display(
    recommendations["confidence"].value_counts()
)

print("\nTop 10 priority pages:")
display(
    top_50.sample(10)
)

Ranked recommendation queue
--------------------------------
Total test pages: 91065
Top 50 pages selected: 50

Confidence distribution:


,count
confidence,
Low,64245
High,23439
Medium,3381



Top 10 priority pages:


,client_hash_id,content_hash_id,report_date,model_score,rank,confidence,suggested_action
10,client_73cda7b4e4f265ea,content_5cb10525b94cad46,2025-04-20,1.0,11,High,Review
35,client_73cda7b4e4f265ea,content_047c6f86cc8dfab1,2025-04-17,1.0,36,High,Review
11,client_73cda7b4e4f265ea,content_df053ccb95bd1fc6,2025-04-20,1.0,12,High,Review
33,client_73cda7b4e4f265ea,content_7d9d2c91c5e1b91f,2025-04-26,1.0,34,High,Review
2,client_9958f0a7ae1df715,content_3fa89de2020a2373,2025-04-29,1.0,3,High,Review
36,client_9958f0a7ae1df715,content_c6702cb22a2c4c56,2025-04-17,1.0,37,High,Review
5,client_9958f0a7ae1df715,content_f0f0f41959f01bb6,2025-04-17,1.0,6,High,Review
42,client_73cda7b4e4f265ea,content_8f356875941d3e51,2025-04-20,1.0,43,High,Review
41,client_73cda7b4e4f265ea,content_7e04a1d823d65e1d,2025-04-26,1.0,42,High,Review
30,client_9958f0a7ae1df715,content_3d9ab63974c13aa9,2025-04-20,1.0,31,High,Review


In [92]:
# Add observable reason codes

# Start from the test data
ranked = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "gsc_avg_position"
    ]
].copy()

# Add model score
ranked["model_score"] = rf_model.predict_proba(X_test)[:, 1]

# Rank by model score
ranked = ranked.sort_values(
    "model_score",
    ascending=False
).reset_index(drop=True)

ranked["rank"] = ranked.index + 1


# Reason codes

def get_reason(row):

    reasons = []

    if row["gsc_impressions"] >= test_df["gsc_impressions"].quantile(0.75):
        reasons.append("high_search_visibility")

    if row["gsc_avg_position"] <= test_df["gsc_avg_position"].quantile(0.25):
        reasons.append("strong_average_position")

    if row["gsc_clicks"] > 0:
        reasons.append("existing_search_clicks")

    if row["gsc_sum_position"] > test_df["gsc_sum_position"].median():
        reasons.append("ranking_signal_present")

    if not reasons:
        reasons.append("model_priority")

    return ", ".join(reasons)


ranked["reason_codes"] = ranked.apply(
    get_reason,
    axis=1
)


# Confidence

def confidence_label(score):

    if score >= 0.80:
        return "High"
    elif score >= 0.60:
        return "Medium"
    else:
        return "Low"


ranked["confidence"] = ranked["model_score"].apply(
    confidence_label
)


# Suggested action

def suggested_action(row):

    if row["model_score"] >= 0.80:
        return "Prioritize review"

    elif row["model_score"] >= 0.60:
        return "Review"

    else:
        return "Monitor"


ranked["suggested_action"] = ranked.apply(
    suggested_action,
    axis=1
)


# Display top 20
display(
    ranked.head(20)[
        [
            "rank",
            "content_hash_id",
            "report_date",
            "model_score",
            "confidence",
            "suggested_action",
            "reason_codes"
        ]
    ]
)

,rank,content_hash_id,report_date,model_score,confidence,suggested_action,reason_codes
0,1,content_88d500a55bd9b986,2025-04-29,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
1,2,content_3fc3b8238a5b1ddf,2025-04-29,1.0,High,Prioritize review,"high_search_visibility, strong_average_positio..."
2,3,content_3fa89de2020a2373,2025-04-29,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
3,4,content_6d1a5f4347b19979,2025-04-29,1.0,High,Prioritize review,"high_search_visibility, strong_average_positio..."
4,5,content_f18b70df51904723,2025-04-20,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
5,6,content_f0f0f41959f01bb6,2025-04-17,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
6,7,content_04441d837222c714,2025-04-17,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
7,8,content_f0d9fb91e7681e85,2025-04-17,1.0,High,Prioritize review,"high_search_visibility, strong_average_positio..."
8,9,content_05554cfab3e9cec8,2025-04-17,1.0,High,Prioritize review,"high_search_visibility, ranking_signal_present"
9,10,content_9d8204e19b49515e,2025-04-20,1.0,High,Prioritize review,"high_search_visibility, strong_average_positio..."


In [93]:
# Recommendation summary

print("Recommendation Summary")
print("----------------------")

print("Total ranked pages:", len(ranked))

print("\nTop 20 pages:")
display(ranked.head(20))

print("\nConfidence distribution:")
display(
    ranked["confidence"].value_counts()
)

print("\nSuggested action distribution:")
display(
    ranked["suggested_action"].value_counts()
)

Recommendation Summary
----------------------
Total ranked pages: 91065

Top 20 pages:


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position,gsc_avg_position,model_score,rank,reason_codes,confidence,suggested_action
0,client_9958f0a7ae1df715,content_88d500a55bd9b986,2025-04-29,143,0,4032.0,28.195804,1.0,1,"high_search_visibility, ranking_signal_present",High,Prioritize review
1,client_9958f0a7ae1df715,content_3fc3b8238a5b1ddf,2025-04-29,73,0,414.0,5.671233,1.0,2,"high_search_visibility, strong_average_positio...",High,Prioritize review
2,client_9958f0a7ae1df715,content_3fa89de2020a2373,2025-04-29,105,0,1107.0,10.542857,1.0,3,"high_search_visibility, ranking_signal_present",High,Prioritize review
3,client_9958f0a7ae1df715,content_6d1a5f4347b19979,2025-04-29,83,0,667.0,8.036145,1.0,4,"high_search_visibility, strong_average_positio...",High,Prioritize review
4,client_73cda7b4e4f265ea,content_f18b70df51904723,2025-04-20,55,0,1070.0,19.454545,1.0,5,"high_search_visibility, ranking_signal_present",High,Prioritize review
5,client_9958f0a7ae1df715,content_f0f0f41959f01bb6,2025-04-17,55,0,2248.0,40.872727,1.0,6,"high_search_visibility, ranking_signal_present",High,Prioritize review
6,client_73cda7b4e4f265ea,content_04441d837222c714,2025-04-17,86,0,4997.0,58.104651,1.0,7,"high_search_visibility, ranking_signal_present",High,Prioritize review
7,client_9958f0a7ae1df715,content_f0d9fb91e7681e85,2025-04-17,76,2,442.0,5.815789,1.0,8,"high_search_visibility, strong_average_positio...",High,Prioritize review
8,client_73cda7b4e4f265ea,content_05554cfab3e9cec8,2025-04-17,145,0,1382.0,9.531034,1.0,9,"high_search_visibility, ranking_signal_present",High,Prioritize review
9,client_73cda7b4e4f265ea,content_9d8204e19b49515e,2025-04-20,267,1,2338.0,8.756554,1.0,10,"high_search_visibility, strong_average_positio...",High,Prioritize review



Confidence distribution:


,count
confidence,
Low,64245
High,23439
Medium,3381



Suggested action distribution:


,count
suggested_action,
Monitor,64245
Prioritize review,23439
Review,3381


### Interpretation

The ranked queue prioritizes pages using the Random Forest model's predicted probability of reaching the selected future-impression threshold.

The highest-ranked pages should be reviewed first because the model assigns them the strongest predicted probability based on information available at prediction time.

Reason codes summarize observable search-performance signals associated with each recommendation. They are descriptive rather than causal.

The queue is intended to support human review and prioritization. A high model score does not prove that refreshing a page will cause future performance to improve.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

==>. Artifacts the paper embeds

The final paper will include the following artifacts generated from the analysis:

1. **Model performance table**
   - Baseline vs Random Forest
   - Accuracy
   - Precision
   - Recall
   - F1 score
   - ROC-AUC
   - Average Precision

2. **Feature importance table**
   - Top features identified by the Random Forest model.
   - The strongest observed signals are Google Search Console impressions,
     total position, average position, and clicks.

3. **Precision@50 comparison**
   - Baseline Precision@50
   - Random Forest Precision@50
   - This represents the quality of the top 50 pages selected for review.

4. **Ranked review output**
   - Pages can be ranked using the model's predicted probability.
   - Higher scores indicate higher priority for human review.

All artifacts use aggregated or pseudonymized information and do not expose
client names, domains, URLs, private queries, or credentials.

In [95]:
# SECTION 7 — RESULTS / ARTIFACTS

results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC",
        "Average Precision"
    ],
    "Baseline": [
        baseline_accuracy,
        baseline_precision,
        baseline_recall,
        baseline_f1,
        baseline_auc,
        baseline_ap
    ],
    "Random Forest": [
        rf_accuracy,
        rf_precision,
        rf_recall,
        rf_f1,
        rf_auc,
        rf_ap
    ]
})

results = results.round(4)

display(results)

,Metric,Baseline,Random Forest
0,Accuracy,0.6539,0.8819
1,Precision,0.0000,0.8624
2,Recall,0.0000,0.7836
3,F1,0.0000,0.8211
4,ROC-AUC,0.5000,0.9276
5,Average Precision,0.3461,0.8954


In [96]:
# Feature importance artifact

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(
    feature_importance.head(15).round(4)
)

,Feature,Importance
0,gsc_impressions,0.6270
1,gsc_sum_position,0.2186
2,gsc_avg_position,0.1293
3,gsc_clicks,0.0251


In [97]:
# Precision@K artifact

precision_20 = precision_at_k(
    y_test.values,
    rf_prob,
    20
)

precision_50 = precision_at_k(
    y_test.values,
    rf_prob,
    50
)

precision_100 = precision_at_k(
    y_test.values,
    rf_prob,
    100
)

precision_k_results = pd.DataFrame({
    "K": [20, 50, 100],
    "Random Forest Precision": [
        precision_20,
        precision_50,
        precision_100
    ]
})

display(
    precision_k_results.round(4)
)

,K,Random Forest Precision
0,20,0.90
1,50,0.94
2,100,0.96


### Results

The Random Forest model substantially outperformed the majority-class baseline on the held-out test period.

The Random Forest achieved a ROC-AUC of 0.9276 and an Average Precision of 0.8954, indicating strong ranking performance on the test data.

The model achieved 0.8624 precision, 0.7836 recall, and an F1 score of 0.8211.

The baseline achieved a ROC-AUC of 0.5000 and Average Precision of 0.3461, providing a substantially weaker reference point.

These results indicate that the current-period features contain useful predictive information for identifying observations likely to reach the selected next-day GSC-impressions threshold.

However, these results should be interpreted as predictive performance rather than evidence of a causal relationship between the observed features and future search performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
